### Setup
As per https://github.com/state-spaces/mamba/tree/main?tab=readme-ov-file

In [ ]:
%pip install einops transformers datasets python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, sys

# Clone repo if running on Colab
if "google.colab" in sys.modules:
    if not os.path.exists("csci739-project-mamba"):
        os.system("git clone https://github.com/JackMclaughlin424/csci739-project-mamba.git")
    else:
        os.system("cd csci739-project-mamba && git pull")
    sys.path.insert(0, "csci739-project-mamba")

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from datasets import load_dataset


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

### Init model

In [ ]:
from mamba.mamba_llm import MambaLMConfig, MambaLMHeadModel

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

config = MambaLMConfig(
    d_input=256,
    d_model=512,     # 2x d_input
    d_state=16,
    n_layer=6,
    vocab_size=len(tokenizer),
    kernel_size=4,
    bias=False,
    conv_bias=True,
    # dt_rank is auto-computed as ceil(d_input / 16) = 16
)

model = MambaLMHeadModel(config).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

### Simple Stories

In [ ]:
from datasets import load_dataset

ds = load_dataset("SimpleStories/SimpleStories")